In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split



data = load_breast_cancer()

X , y = data.data , data.target # X the entire datasets , y ==> target which is 1 malignant 0 = benign 

labels = data.target_names # beingn or malignant 

feature_names = data.feature_names # names of each  feature


np.random.seed(42) # ensures every time I get a similar result 

# splitting before traing avoids data leakage 
X_train, X_test , y_train , y_test = train_test_split(X , y , test_size= 0.7 , random_state= 42 , stratify=y)

scale = StandardScaler() # mean = 0 and std  = 1 for numerical values
X_train_scaled = scale.fit_transform(X_train) # ? why wouldm't I fit and transform y
X_test_scaled = scale.transform(X_test) # y ==> 0 and 1 there is no mean or std here , we can't use StandarSCaler for numerical values


df_train = pd.DataFrame(X_train_scaled, columns=feature_names)
df_test = pd.DataFrame(X_test_scaled, columns=feature_names)

noise_factor = 0.5 

noise = noise_factor * np.random.normal(loc=0, scale=1 , size=X_train_scaled.shape)
X_train_noisy = noise + X_train_scaled
df_noisy_X = pd.DataFrame(X_train_noisy, columns=feature_names)

# Visulization starts here 👇👇


# Two histograms side-by-side + scatter plot + line plot

fig , axes = plt.subplots(2,2,figsize=(14,8))

col_index = 5

col_name = feature_names[col_index]

axes[0,0].hist(X_train_scaled[:,col_index], bins=20 , alpha= 0.7)
axes[0,0].set_title("Original Feature Distribution")
axes[0,0].set_xlabel(col_name)
axes[0,0].set_ylabel("Frequency")
axes[0,0].grid(True)

axes[0,1].hist(df_noisy_X[col_name], bins=20 , alpha= 0.7)
axes[0,1].set_title("Noisy Feature Distribution")
axes[0,1].set_xlabel(col_name)
axes[0,1].set_ylabel("Feature value")
axes[0,1].grid(True)


axes[1,0].scatter(X_train_scaled[:,col_index],X_train_noisy[:,col_index],s=10, alpha=0.7)
axes[1,0].set_title("Scatter plot of the data")
axes[1,0].set_xlabel(col_name)
axes[1,0].set_ylabel("Feature value")
axes[1,0].grid(True)

axes[1,1].plot(X_train_scaled[:,col_index],lw=2,label="Original")
axes[1,1].plot(X_train_noisy[:,col_index],"--", lw=1,label="Noisy")
axes[1,1].set_title("Comparison of the dataset")
axes[1,1].set_xlabel(col_name)
axes[1,1].set_ylabel("Frequency")
axes[1,1].grid(True)

plt.tight_layout()
plt.show()


# Here we create Knn model

Knn = KNeighborsClassifier(n_neighbors=5)
Knn_model_fit = Knn.fit( X_train_scaled, y_train)
Knn_model_prediction = Knn.predict(X_test_scaled)
# Here we create SVM model
svm= SVC(kernel="linear", C=1.0 , random_state=42)
svm_model_fit = svm.fit(X_train_scaled, y_train)
svm_model_prediction = svm.predict(X_test_scaled)


# Evaluate Classification report and accuracy of KNN
knn_accuracy = accuracy_score(y_test, Knn_model_prediction)
knn_report = classification_report(y_test, Knn_model_prediction)

print(f"KNN Accuracy: {knn_accuracy:.4f}\n{knn_report}")

# Evaluate Classification report and accuracy of SVM

svm_accuracy = accuracy_score(y_test, svm_model_prediction)
svm_report = classification_report(y_test, svm_model_prediction)

print(f"SVM Accuracy: {svm_accuracy:.4f}\n{svm_report}")

# Create a confusion matrix:

Knn_confusion_matrix = confusion_matrix(y_test , Knn_model_prediction)
svm_confusion_matrix = confusion_matrix(y_test , svm_model_prediction)

fig , axes = plt.subplots(1,2,figsize=(14,6))

sns.heatmap(Knn_confusion_matrix,
            cmap="Blues",
            annot=True,
            fmt = "d",
            xticklabels=labels,
            yticklabels=labels,
            ax= axes[0],)
axes[0].set_title("Knn confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
sns.heatmap(svm_confusion_matrix,
            cmap="Blues",
            fmt="d",
            annot=True,
            xticklabels=labels,
            yticklabels=labels,
            ax=axes[1])
axes[1].set_title("SVM confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()
